In [5]:
import pandas as pd
import sys
from pathlib import Path
import asyncio
import time

sys.path.append(str(Path().resolve().parent.parent))
from src.gradient_client import GradientSportsClient

pd.set_option('display.max_columns', None)

### Status da requisição

In [6]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

In [7]:
games = pd.read_csv(str(Path().resolve().parent.parent / "data" / "games.csv"))

# remover jogos da temporada atual pois não serão usados
games = games[~games['season'].isin(['2025-2026', '2026'])].reset_index(drop=True)
games

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
0,4447,2022-08-13,2022-2023,Right,Left,TEAM_HOME,3,Aston Villa,1,Premier League,8,Everton,Villa Park,105.0,68.0
1,4760,2023-04-25,2022-2023,Left,Left,OPPONENT_HOME,7,Crystal Palace,1,Premier League,20,Wolverhampton Wanderers,Molineux,105.0,68.0
2,12806,2023-11-04,2023,Left,Right,OPPONENT_HOME,517,Bahia,42,Brasileiro Série A,515,Grêmio,Arena do Grêmio,105.0,68.0
3,32206,2025-01-18,2024-2025,Right,Left,TEAM_HOME,119,Brentford,1,Premier League,10,Liverpool,Gtech Community Stadium,105.0,68.0
4,4451,2022-08-15,2022-2023,Left,Left,OPPONENT_HOME,7,Crystal Palace,1,Premier League,10,Liverpool,Anfield,101.0,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3021,231,2020-11-21,2020-2021,Right,Left,TEAM_HOME,3,Aston Villa,1,Premier League,4,Brighton & Hove Albion,Villa Park,105.0,68.0
3022,4660,2023-02-12,2022-2023,Right,Left,TEAM_HOME,55,Leeds United,1,Premier League,12,Manchester United,Elland Road,105.0,68.0
3023,1224,2021-12-04,2021-2022,Left,Left,OPPONENT_HOME,10,Liverpool,1,Premier League,20,Wolverhampton Wanderers,Molineux,105.0,68.0
3024,12744,2023-09-30,2023,Right,Right,TEAM_HOME,437,Fortaleza,42,Brasileiro Série A,515,Grêmio,Arena Castelao,105.0,68.0


In [ ]:
SEM_LIMIT = 5
semaphore = asyncio.Semaphore(SEM_LIMIT)

# Cria a pasta "events" caso não exista
events_dir = Path().resolve().parent.parent / "data" / "events"
events_dir.mkdir(parents=True, exist_ok=True)  # Cria a pasta se não existir

async def fetch_game_events(game, index, total):
    game_id = game['id']
    game_season = game['season']
    game_competition = game['competition.id']
    
    async with semaphore:
        print(f"[{index + 1} / {total}] Starting request for game: {game_id}, competition id: {game_competition}, season: {game_season}")
        start_time = time.perf_counter()
        
        await asyncio.sleep(0.2)  # Controle para não estourar API
        
        df = await asyncio.to_thread(
            client.get_game_events_flat,
            game_id,
            as_dataframe=True
        )
        
        elapsed = time.perf_counter() - start_time
        print(f"[{index + 1} / {total}] Finished request for game {game_id} in {elapsed:.2f} seconds")
    
    df['competitionId'] = game_competition
    df['gameId'] = game_id
    df['season'] = game_season
    return df, game_competition, game_id, game_season

async def process_group(group_df):
    games_list = [row for _, row in group_df.head().iterrows()]
    total = len(games_list)
    
    tasks = [
        asyncio.create_task(fetch_game_events(game, index, total))
        for index, game in enumerate(games_list)
    ]
    
    for task in asyncio.as_completed(tasks):
        df, competition_id, game_id, season = await task
        
        events_dir = Path().resolve().parent.parent / "data" / "events"/ f"{competition_id}" / f"{season}"
        events_dir.mkdir(parents=True, exist_ok=True)  # Cria a pasta se não existir

        file_name = f"events_{game_id}.parquet"
        df.to_parquet(str(Path().resolve().parent.parent / "data" / "events" / f"{competition_id}" / f"{season}" /f"{file_name}"), index=False)
        print(f"Saved {file_name} with {len(df)} rows")
       
    return df

async def main():
    # Garanta que a coluna 'competition.id' e 'season' existam no DataFrame games
    grouped = games.groupby(['competition.id', 'season'])
    
    for (competition, season), group_df in grouped:
        print(f"\nProcessing competition id {competition}, season {season} - {len(group_df)} games")
        
        df_group = await process_group(group_df)
        
        print(f"Process completed for games of competition id {competition} and season {season}")
        
        # Limpar memória antes de próxima iteração (se necessário)
        del df_group

await main()


Processing competition id 1, season 2020-2021 - 378 games
[1 / 5] Starting request for game: 659, competition id: 1, season: 2020-2021
[2 / 5] Starting request for game: 287, competition id: 1, season: 2020-2021
[3 / 5] Starting request for game: 391, competition id: 1, season: 2020-2021
[4 / 5] Starting request for game: 414, competition id: 1, season: 2020-2021
[5 / 5] Starting request for game: 386, competition id: 1, season: 2020-2021
[5 / 5] Finished request for game 386 in 9.96 seconds
Saved events_386.csv with 2549 rows
[4 / 5] Finished request for game 414 in 12.45 seconds
Saved events_414.csv with 2875 rows
[1 / 5] Finished request for game 659 in 14.61 seconds
Saved events_659.csv with 2622 rows
[3 / 5] Finished request for game 391 in 15.17 seconds
Saved events_391.csv with 2645 rows
[2 / 5] Finished request for game 287 in 15.88 seconds
Saved events_287.csv with 2555 rows
Process completed for games of competition id 1 and season 2020-2021

Processing competition id 1, sea